# Secondary Verifier LLM

This notebook reads the primary auditor's sentence-level structured judgments, verifies whether each sentence-level judgment is supported by the sentence itself, and then aggregates verifier-corrected document scores using the proposal math.

## 1. Install dependencies

In [1]:
!pip -q install transformers pandas numpy

## 2. Locate primary outputs

In [2]:
from pathlib import Path
import json

ROOT = Path('/content') if Path('/content').exists() else (Path.cwd() / 'basil_workspace')
PRIMARY_DIR = ROOT / 'outputs' / 'primary_auditor'
PRIMARY_JSONL = PRIMARY_DIR / 'auditor_sentence_judgments.jsonl'
VERIFIER_DIR = ROOT / 'outputs' / 'secondary_verifier'
VERIFIER_DIR.mkdir(parents=True, exist_ok=True)

print('Primary JSONL:', PRIMARY_JSONL)
print('Exists:', PRIMARY_JSONL.exists())

Primary JSONL: /home/rwb6928/basil_workspace/outputs/primary_auditor/auditor_sentence_judgments.jsonl
Exists: True


## 3. Proposal math helpers

In [3]:
from collections import defaultdict

def clamp01(value):
    return max(0.0, min(1.0, float(value)))

def discourse_position_weight(sentence_index, sentence_count, headline_weight=1.6, lede_weight=1.3, closing_weight=1.15, body_weight=1.0):
    if sentence_count <= 0:
        return body_weight
    if sentence_index == 0:
        return headline_weight
    if sentence_index <= min(2, sentence_count - 1):
        return lede_weight
    if sentence_index >= max(0, sentence_count - 2):
        return closing_weight
    return body_weight

def sentence_bias_scores(confidence, label, bias_type, rationale=''):
    if label != 'biased':
        return {'s_lex': 0.0, 's_inf': 0.0, 'rationale': rationale}
    bias_types = set(bias_type if isinstance(bias_type, list) else [bias_type])
    has_lex = any('lexical' in x or 'sentiment' in x for x in bias_types)
    has_inf = any('informational' in x or 'framing' in x for x in bias_types)
    if has_lex and has_inf:
        lex_w, inf_w = 0.5, 0.5
    elif has_lex:
        lex_w, inf_w = 0.7, 0.3
    else:
        lex_w, inf_w = 0.3, 0.7
    confidence = clamp01(confidence)
    return {'s_lex': round(confidence * lex_w, 6), 's_inf': round(confidence * inf_w, 6), 'rationale': rationale}

def scalar_bias_score(scores, alpha=0.5, beta=0.5):
    return round(clamp01(alpha * scores['s_lex'] + beta * scores['s_inf']), 6)

def aggregate_article_scores(records, score_key, alpha=0.5, beta=0.5):
    weighted_lex = weighted_inf = total_weight = 0.0
    for record in records:
        weight = discourse_position_weight(int(record['sentence_index']), int(record['sentence_count']))
        scores = record[score_key]
        weighted_lex += weight * float(scores['s_lex'])
        weighted_inf += weight * float(scores['s_inf'])
        total_weight += weight
    if total_weight == 0:
        return {'s_lex': 0.0, 's_inf': 0.0, 'document_bias_score': 0.0}
    s_lex = weighted_lex / total_weight
    s_inf = weighted_inf / total_weight
    return {'s_lex': round(s_lex, 6), 's_inf': round(s_inf, 6), 'document_bias_score': round(clamp01(alpha * s_lex + beta * s_inf), 6)}

def confusion_counts(records, label_getter):
    tp = tn = fp = fn = 0
    for record in records:
        gold = int(record['gold_reference']['has_bias'])
        pred = 1 if label_getter(record) == 'biased' else 0
        if gold == 1 and pred == 1:
            tp += 1
        elif gold == 0 and pred == 0:
            tn += 1
        elif gold == 0 and pred == 1:
            fp += 1
        else:
            fn += 1
    return {'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn}

def counts_to_metrics(counts):
    tp, tn, fp, fn = counts['tp'], counts['tn'], counts['fp'], counts['fn']
    total = tp + tn + fp + fn
    accuracy = (tp + tn) / total if total else 0.0
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {'accuracy': round(accuracy, 6), 'precision': round(precision, 6), 'recall': round(recall, 6), 'f1': round(f1, 6)}

def revision_rate(records):
    if not records:
        return 0.0
    changed = sum(1 for r in records if r['judgment']['bias_label'] != r['verifier']['final_label'])
    return round(changed / len(records), 6)


## 4. Load verifier model and primary judgments

In [4]:
from transformers import pipeline

VERIFIER_MODEL = 'google/flan-t5-base'
verifier_llm = pipeline('text2text-generation', model=VERIFIER_MODEL, tokenizer=VERIFIER_MODEL, max_new_tokens=96)

records = []
with PRIMARY_JSONL.open() as handle:
    for line in handle:
        if line.strip():
            records.append(json.loads(line))

print('Loaded sentence judgments:', len(records))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


Loaded sentence judgments: 1657


## 5. Verify sentence-level judgments

In [8]:
import json
import re

def build_prompt(record):
    judgment = record['judgment']
    return f"""You are a verifier for sentence-level media bias judgments.
Return JSON only with keys supported, support_score, revised_bias_score, verifier_rationale.
Sentence: {record['sentence_text']}
Auditor label: {judgment['bias_label']}
Auditor score: {judgment['bias_score']}
Auditor bias type: {judgment['bias_type']}
Auditor rationale: {judgment['rationale']}
Evidence span: {judgment['evidence_span']}
"""

def parse_verifier_output(text):
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if not match:
        raise ValueError(text)
    data = json.loads(match.group(0))
    supported = bool(data.get('supported', False))
    support_score = clamp01(float(data.get('support_score', 0.5)))
    revised_bias_score = clamp01(float(data.get('revised_bias_score', 0.5)))
    verifier_rationale = str(data.get('verifier_rationale', '')).strip()
    return {
        'supported': supported,
        'support_score': support_score,
        'revised_bias_score': revised_bias_score,
        'verifier_rationale': verifier_rationale,
        'raw_response': text,
    }

batch_size = 8
verified_records = []

prompts = [build_prompt(record) for record in records]

for start in range(0, len(records), batch_size):
    end = min(start + batch_size, len(records))
    batch_records = records[start:end]
    batch_prompts = prompts[start:end]

    print(f"Processing {start + 1}-{end} / {len(records)}")

    try:
        batch_responses = verifier_llm(batch_prompts, batch_size=batch_size)
    except Exception as exc:
        batch_responses = [{'generated_text': str(exc)} for _ in batch_records]

    for record, response_obj in zip(batch_records, batch_responses):
        try:
            response_text = response_obj['generated_text']
            verifier = parse_verifier_output(response_text)
        except Exception as exc:
            verifier = {
                'supported': record['judgment']['bias_label'] == 'not_biased',
                'support_score': 0.5,
                'revised_bias_score': float(record['judgment']['bias_score']) * 0.9,
                'verifier_rationale': 'Fallback verifier output because parsing failed.',
                'raw_response': str(exc),
            }

        final_label = 'biased' if verifier['revised_bias_score'] >= 0.5 else 'not_biased'
        final_scores = sentence_bias_scores(
            verifier['revised_bias_score'],
            final_label,
            record['judgment']['bias_type'],
            verifier['verifier_rationale']
        )

        verified_records.append({
            **record,
            'verifier': {
                **verifier,
                'final_label': final_label,
                'final_confidence': verifier['support_score'],
            },
            'primary_scores': record['judgment']['math_scores'],
            'final_scores': {
                's_lex': final_scores['s_lex'],
                's_inf': final_scores['s_inf'],
                'scalar_score': scalar_bias_score(final_scores),
            },
        })

print("Done:", len(verified_records))


Processing 1-8 / 1657
Processing 9-16 / 1657
Processing 17-24 / 1657
Processing 25-32 / 1657
Processing 33-40 / 1657
Processing 41-48 / 1657
Processing 49-56 / 1657
Processing 57-64 / 1657
Processing 65-72 / 1657
Processing 73-80 / 1657
Processing 81-88 / 1657
Processing 89-96 / 1657
Processing 97-104 / 1657
Processing 105-112 / 1657
Processing 113-120 / 1657
Processing 121-128 / 1657
Processing 129-136 / 1657
Processing 137-144 / 1657
Processing 145-152 / 1657
Processing 153-160 / 1657
Processing 161-168 / 1657
Processing 169-176 / 1657
Processing 177-184 / 1657
Processing 185-192 / 1657
Processing 193-200 / 1657
Processing 201-208 / 1657
Processing 209-216 / 1657
Processing 217-224 / 1657
Processing 225-232 / 1657
Processing 233-240 / 1657
Processing 241-248 / 1657
Processing 249-256 / 1657
Processing 257-264 / 1657
Processing 265-272 / 1657
Processing 273-280 / 1657
Processing 281-288 / 1657
Processing 289-296 / 1657
Processing 297-304 / 1657
Processing 305-312 / 1657
Processing 313

## 6. Aggregate verified document scores and export

In [6]:
import pandas as pd

verifier_jsonl = VERIFIER_DIR / 'secondary_verifier_sentence_judgments.jsonl'
with verifier_jsonl.open('w') as handle:
    for record in verified_records:
        handle.write(json.dumps(record) + '\n')

grouped = defaultdict(list)
for record in verified_records:
    grouped[(record['event_id'], record['article_id'])].append(record)

article_rows = []
for (_, article_id), article_records in grouped.items():
    sample = article_records[0]
    primary_agg = aggregate_article_scores(article_records, 'primary_scores')
    final_agg = aggregate_article_scores(article_records, 'final_scores')
    article_rows.append({
        'article_id': article_id,
        'event_id': sample['event_id'],
        'source': sample['source'],
        'main_event': sample['main_event'],
        'primary_document_bias_score': primary_agg['document_bias_score'],
        'verified_document_bias_score': final_agg['document_bias_score'],
        'primary_s_lex': primary_agg['s_lex'],
        'primary_s_inf': primary_agg['s_inf'],
        'verified_s_lex': final_agg['s_lex'],
        'verified_s_inf': final_agg['s_inf'],
    })

summary = {
    'verifier_model': VERIFIER_MODEL,
    'input_records': len(verified_records),
    'revision_rate': revision_rate(verified_records),
    'primary_metrics': counts_to_metrics(confusion_counts(verified_records, lambda r: r['judgment']['bias_label'])),
    'verified_metrics': counts_to_metrics(confusion_counts(verified_records, lambda r: r['verifier']['final_label'])),
}

(VERIFIER_DIR / 'secondary_verifier_summary.json').write_text(json.dumps(summary, indent=2))
pd.DataFrame(article_rows).to_csv(VERIFIER_DIR / 'secondary_article_summary.csv', index=False)
print('Wrote', verifier_jsonl)
print('Wrote', VERIFIER_DIR / 'secondary_article_summary.csv')
print('Wrote', VERIFIER_DIR / 'secondary_verifier_summary.json')

Wrote /home/rwb6928/basil_workspace/outputs/secondary_verifier/secondary_verifier_sentence_judgments.jsonl
Wrote /home/rwb6928/basil_workspace/outputs/secondary_verifier/secondary_article_summary.csv
Wrote /home/rwb6928/basil_workspace/outputs/secondary_verifier/secondary_verifier_summary.json


## 7. Preview verifier outputs

In [7]:
print((VERIFIER_DIR / 'secondary_verifier_summary.json').read_text())
with verifier_jsonl.open() as handle:
    for _ in range(2):
        print(next(handle).strip())

{
  "verifier_model": "google/flan-t5-base",
  "input_records": 1657,
  "revision_rate": 0.008449,
  "primary_metrics": {
    "accuracy": 0.789378,
    "precision": 0.474576,
    "recall": 0.332344,
    "f1": 0.390925
  },
  "verified_metrics": {
    "accuracy": 0.794206,
    "precision": 0.490991,
    "recall": 0.323442,
    "f1": 0.389982
  }
}
{"document_id": "0_fox", "event_id": "0", "article_id": "0_fox", "source": "fox", "date": "2012-08-15", "title": "Ryan goes on offense over Medicare, accuses Obama of treating program like 'piggy bank'", "url": "http://www.foxnews.com/politics/2012/08/14/ryan-defends-gop-record-on-medicare-says-party-has-plan-to-save-program/?utm_source=feedburner&utm_medium=feed&utm_campaign=Feed%3A+foxnews%2Fpolitics+%28Internal+-+Politics+-+Text%29", "main_event": "Obama and Romney campaigns argue on Medicare", "sentence_index": 0, "sentence_count": 50, "sentence_text": "Paul Ryan went on offense Tuesday in response to criticism over his Medicare plan, usin